# Пакетная сборка корпуса из нескольких master-review `.ipynb`

Загрузите в Colab **несколько ноутбуков** одного проекта (итерации ревью). В ячейках должны быть маркеры:
- `Комментарий ревьюера`
- `Комментарий мидл-ревьюера` или `Комментарий мидл ревьюера`
- `Комментарий студента`

**Runtime JSONL** (для `PROJECT_REVIEW_TRAINING_PATH`) содержит только реплики ревьюера и мидл-ревьюера. **Fine-tune JSONL** — все роли, включая студента (для внешнего обучения). Ничего из этого не является ground truth для движка.

После скачивания файлов задайте в `.env` сервиса: `ENABLE_RETRIEVAL=true`, `ENABLE_PROJECT_REVIEW_TRAINING=true`, `PROJECT_REVIEW_TRAINING_PATH=...`

In [ ]:
# !pip -q install nbformat

import json
import os
from pathlib import Path

import nbformat
# from google.colab import files

# В Colab: положите .ipynb в папку UPLOAD_DIR (или используйте files.upload() вручную)
UPLOAD_DIR = Path("master_review_ipynb")
UPLOAD_DIR.mkdir(exist_ok=True)
SOURCE_PROJECT = "my_project_slug"
RUNTIME_JSONL = Path("project_training.jsonl")
FINETUNE_JSONL = Path("finetune_dialogue.jsonl")

# Раскомментируйте для загрузки с машины:
# uploaded = files.upload()
# for name in uploaded:
#     Path(name).rename(UPLOAD_DIR / name)

import sys
if "review_assistant_repo" not in str(Path.cwd()):
    print("Склонируйте репозиторий review assistant и добавьте app/ в PYTHONPATH, либо скопируйте app/retrieval/notebook_training.py")

sys.path.insert(0, ".")
try:
    from app.retrieval.notebook_training import extract_rows_from_ipynb, merge_write_jsonl
except ImportError:
    raise ImportError("Добавьте корень пакета (где лежит app/) в PYTHONPATH")

runtime_all, finetune_all = [], []
for ipynb in sorted(UPLOAD_DIR.glob("*.ipynb")):
    r, f = extract_rows_from_ipynb(ipynb, source_project=SOURCE_PROJECT)
    runtime_all.extend(r)
    finetune_all.extend(f)

merge_write_jsonl(runtime_all, RUNTIME_JSONL)
merge_write_jsonl(finetune_all, FINETUNE_JSONL)
print(f"Runtime: {len(runtime_all)} -> {RUNTIME_JSONL}")
print(f"Finetune: {len(finetune_all)} -> {FINETUNE_JSONL}")